<a href="https://colab.research.google.com/github/katelynnlindsey/phonology-chuvash/blob/main/logbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Acousting analysis of Chuvash vowels

1. Obtain acoustic data

2. Pre-process for Montreal Forced Aligner (MFA)

3. Run MFA

4. Pre-process for FAVE vowel extraction

5. Run new-FAVE

In [ ]:
!pip install new-fave

6. Compile extracted vowel data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# --- Define Path to the COMBINED CSV ---
# VERY IMPORTANT: Double-check this string for ANY typos.
base_corpus_path_str = "C:\\Users\\profk\\Documents\\GitHub\\phonology-chuvash\\corpora"
combined_csv_filename = "all_chuvash_vowel_points.csv"

# Ensure the path is fully normalized to the OS's native format
combined_csv_path = os.path.normpath(os.path.join(base_corpus_path_str, combined_csv_filename))

# Initialize vowels_df to an empty DataFrame, so it's always defined
vowels_df = pd.DataFrame()

# --- CRITICAL DIAGNOSTIC BLOCK ---
print(f"--- Jupyter Path Diagnostic ---")
print(f"Jupyter's Current Working Directory (CWD): '{os.getcwd()}'")
print(f"Absolute path to base_corpus_path: '{base_corpus_path_str}'")

if os.path.exists(base_corpus_path_str):
    print(f"CONFIRM: Base corpus directory '{base_corpus_path_str}' DOES exist.")
    # If the base directory exists, let's list its contents
    print(f"CONFIRM: Listing contents of '{base_corpus_path_str}':")
    try:
        dir_contents = os.listdir(base_corpus_path_str)
        for item in dir_contents:
            print(f"         - '{item}'")
        if combined_csv_filename in dir_contents:
            print(f"CONFIRM: '{combined_csv_filename}' IS found directly in the base corpus directory.")
        else:
            print(f"WARNING: '{combined_csv_filename}' is NOT found in the base corpus directory. Check filename or location.")
    except Exception as e:
        print(f"ERROR: Could not list contents of '{base_corpus_path_str}': {e}", file=sys.stderr)
else:
    print(f"ERROR: Base corpus directory '{base_corpus_path_str}' DOES NOT exist. This is the root cause.", file=sys.stderr)
    print(f"ACTION: Double-check the string in 'base_corpus_path_str' for typos or ensure the directory exists at that exact location.", file=sys.stderr)
    # If the base path itself doesn't exist, we can't proceed.
    sys.exit(1) # Exit the script or stop execution if running in a notebook

print(f"Calculated full CSV path for loading: '{combined_csv_path}'")
if os.path.exists(combined_csv_path):
    print(f"CONFIRM: Full CSV file '{combined_csv_path}' DOES exist and should be loadable.")
else:
    print(f"ERROR: Full CSV file '{combined_csv_path}' DOES NOT exist. This is unexpected after previous checks.", file=sys.stderr)
    sys.exit(1) # Exit if the file doesn't exist after all checks
print(f"--- End Diagnostic ---")


# --- Load the combined data ONCE ---
try:
    print(f"Attempting to load combined data from FINAL path: '{combined_csv_path}'")
    combined_df = pd.read_csv(combined_csv_path, encoding='utf-8')
    print(f"Successfully loaded combined data with {len(combined_df)} entries.")
    print("Columns available:", combined_df.columns.tolist())

    # Filter for only vowel labels (exclude SIL and any non-vowel markers)
    vowels_df = combined_df[~combined_df['label'].isin(['SIL', '<eps>', '#'])].copy()
    print(f"Filtered for vowels: {len(vowels_df)} entries remaining.")

except Exception as e: # Catch any remaining exceptions during load/filter
    print(f"An unexpected error occurred during final data loading or filtering: {e}", file=sys.stderr)
    vowels_df = pd.DataFrame()

7. Visualize results

In [ ]:
# Cell 1: Basic Vowel Plot (F2 vs F1)
if not vowels_df.empty:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=vowels_df, x='F2', y='F1', hue='label', alpha=0.6, s=20)
    plt.gca().invert_xaxis()
    plt.gca().invert_yaxis()
    plt.title('Chuvash Vowel Plot (F2 vs F1)')
    plt.xlabel('F2 (Hz)')
    plt.ylabel('F1 (Hz)')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(title='Vowel Label', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print("No vowel data available for F1/F2 plot.")

In [ ]:
if not vowels_df.empty: # Example for next cell
    plt.figure(figsize=(12, 6))
    sns.violinplot(data=vowels_df, x='label', y='F1', inner='quartile', palette='viridis')
    plt.title('F1 Distribution by Vowel')
    plt.xlabel('Vowel Label')
    plt.ylabel('F1 (Hz)')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("No vowel data available for F1 distribution plot.")